# Ollama 설치 및 세팅

In [ ]:
!sudo apt-get update && sudo apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
import time

# 1. Ollama 서버를 백그라운드에서 실행
# stdout/stderr를 무시하지 않고 연결해두면 나중에 로그 확인이 가능합니다.
process = subprocess.Popen(['ollama', 'serve'],
                           stdout=subprocess.PIPE,
                           stderr=subprocess.PIPE,
                           text=True)

# 2. 서버가 완전히 준비될 때까지 대기
print("Ollama 서버를 시작하는 중...")
time.sleep(10)
print("Ollama 서버가 준비되었습니다!")

In [ ]:
!ollama pull gemma3:4b

# RAG 체인 구현

In [ ]:
!pip install langchain-core
!pip install -U langchain langchain-community langchain-huggingface langchain-text-splitters pymupdf sentence-transformers chromadb

## JCCR 논문 기반 문장 단위 청킹 함수 생성

In [ ]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings # 필수: 숫자 변환기
from langchain_community.vectorstores import Chroma      # 필수: 저장소
import re
from langchain_core.documents import Document

# 1. 임베딩 모델 설정 (이건 꼭 있어야 해!)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# 2.청킹 함수
def sliding_window_chunking(docs, window_size=5, step=2):
    """문장 단위로 슬라이딩 윈도우 청킹 수행 (JCCR 논문 전략 반영)"""
    chunked_docs = []

    for doc in docs:
        # 1. 문장 단위로 분리 (마침표, 물음표 등 기준)
        sentences = re.split(r'(?<=[.!?])\s+', doc.page_content)

        # 2. 슬라이딩 윈도우 적용
        for i in range(0, len(sentences), step):
            window = sentences[i : i + window_size]
            if not window: continue

            chunk_content = " ".join(window)

            # 메타데이터 유지 및 청크 정보 추가
            new_metadata = doc.metadata.copy()
            new_metadata["chunk_index"] = i // step

            chunked_docs.append(Document(page_content=chunk_content, metadata=new_metadata))

            # 마지막 문단 처리
            if i + window_size >= len(sentences): break

    return chunked_docs

print("✅ RAG 핵심 엔진(임베딩)과 커스텀 청킹 함수 준비 완료!")

## IPCC 문서 로드 및 청킹

In [ ]:
# 파일 로드 및 청킹 통합 실행
all_processed_chunks = []

# Google Drive 내 IPCC_data 폴더 경로
BASE_PATH = "/content/drive/MyDrive/IPCC_data/"

ipcc_files = [
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_1.pdf", # '세션1.pdf'
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_2.pdf",    # '세션2.pdf'
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_3.pdf",    # '세션3.pdf'
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Section_4.pdf",    # '세션4.pdf'
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Annex_1_Glossary.pdf", # '부속서_용어집.pdf
    BASE_PATH + "IPCC_AR6_SYR_FullVolume_Annex_2_Acronyms.pdf"  # '부속서_약어.pdf'
]

for file_name in ipcc_files:
    if os.path.exists(file_name):
        loader = PyMuPDFLoader(file_name)
        pages = loader.load()

        # 각 파일의 메타데이터 주입
        # 파일명에서 특정 접두사와 .pdf 확장자를 제거하여 섹션 이름을 추출합니다.
        section_name = os.path.basename(file_name).replace("IPCC_AR6_SYR_FullVolume_", "").replace(".pdf", "")
        for p in pages: p.metadata["section"] = section_name

        # 논문 기반 슬라이딩 윈도우 청킹 적용
        chunks = sliding_window_chunking(pages)
        all_processed_chunks.extend(chunks)
        print(f"✅ {section_name}: {len(chunks)}개 청크 생성 완료")
    else:
        print(f"❌ 파일이 존재하지 않습니다: {file_name}. Google Drive에 파일이 올바르게 업로드되었는지 확인해주세요.")

# 벡터 DB 생성
if all_processed_chunks:
    vectorstore = Chroma.from_documents(documents=all_processed_chunks, embedding=embeddings)
    print("🔥 모든 파일이 벡터 데이터베이스에 저장되었습니다!")
else:
    print("⚠️ 처리된 청크가 없어 벡터 데이터베이스를 생성할 수 없습니다.")

## chroma_db에 저장

In [ ]:
# 이전 인덱싱 셀에서 생성한 vectorstore 객체가 있는 상태에서 실행
vectorstore = Chroma.from_documents(
    documents=all_processed_chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"  # 이 경로를 지정해야 파일로 저장됩니다.
)

print("✅ 벡터 DB가 ./chroma_db 폴더에 물리적으로 저장되었습니다.")

## Ollama 검색 함수 생성

In [ ]:
def ask_ollama(prompt, model="gemma3:4b"):
    """
    Ollama API를 호출하여 답변을 생성합니다.
    실패 시 'Failed to generate response'를 반환하여
    app.py의 필터링 로직이 감지할 수 있도록 합니다.
    """
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }

    try:
        response = requests.post(url, json=payload, timeout=90) # 타임아웃 추가로 무한 대기 방지
        response.raise_for_status()

        # .get()을 사용하여 키가 없을 경우의 에러 방지
        result = response.json()
        return result.get('response', "Failed to generate response")

    except Exception as e:
        # 연결 오류나 HTTP 에러 발생 시 지정된 에러 문구 반환
        print(f"Ollama 커뮤니케이션 에러: {e}") # 로그 확인용
        return "Failed to generate response"

In [ ]:
def ipcc_ai_analyst(question, vectorstore): # vectorstore 인자 추가
    # 1. 지식 검색 (상위 4개 조각 추출)
    docs = vectorstore.similarity_search(question, k=4)

    # 2. 컨텍스트 구성 및 출처 정리
    # .get()을 사용하여 메타데이터가 없을 경우 'Unknown'으로 처리
    context_list = []
    source_list = []

    for d in docs:
        section = d.metadata.get('section', 'Unknown')
        context_list.append(f"[Source: {section}]\n{d.page_content}")
        source_list.append(section)

    context = "\n\n".join(context_list)
    sources = list(set(source_list))

    # 3. IPCC 전용 프롬프트 설계
    prompt = f"""
너는 IPCC 6차 보고서(AR6) 전문 분석가야. 아래 제공된 [Context]를 바탕으로 질문에 답해줘.

[Context]
{context}

[Question]
{question}

[Guidelines]
- 반드시 제공된 Context의 내용을 근거로 답변할 것.
- 답변은 전문적이고 신뢰감 있는 한국어로 작성할 것.
- 인류의 활동에 의한 기후 변화의 책임을 명확히 전달할 것 (Action-oriented).
- 전문 용어는 영어 원문을 괄호 안에 병기할 것 (예: 탄소 중립(Net Zero)).
- 답변 마지막에 참고한 섹션들을 나열할 것.

답변:
"""

    # 4. Ollama 호출 (우리가 이전에 만든 ask_ollama 사용)
    response = ask_ollama(prompt)

    # 5. 에러 체크 로직 (정확한 매칭을 위해 전체 문구 사용 권장)
    error_keywords = ["Failed to generate response", "답변 생성 실패", "Ollama 연결 오류", "에러 발생"]
    if any(keyword in response for keyword in error_keywords):
        return response, [], []

    return response, docs, sources

# 쳇봇 구현

In [ ]:
!pip install -q streamlit
!npm install -g localtunnel
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## app.py 생성

In [ ]:
%%writefile app.py
import streamlit as st
import requests
import os
import gc
import fitz
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from PIL import Image
import io

# --- 1. Page Configuration ---
st.set_page_config(page_title="기후 위기 알리미: IPCC AR6 해설봇", page_icon="🌍", layout="wide")

def clear_memory():
    gc.collect()

# --- 2. Resource Loading ---
@st.cache_resource
def load_resources():
    clear_memory()
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    db_path = "./chroma_db"
    if os.path.exists(db_path):
        return Chroma(persist_directory=db_path, embedding_function=embeddings)
    return None

@st.cache_data
def get_pdf_thumbnail(pdf_path, page_num):
    try:
        if not os.path.exists(pdf_path): return None
        doc = fitz.open(pdf_path)
        if page_num < 0 or page_num >= len(doc): return None
        page = doc.load_page(page_num)
        pix = page.get_pixmap(matrix=fitz.Matrix(0.3, 0.3))
        return Image.open(io.BytesIO(pix.tobytes("png")))
    except Exception: return None

def ask_ollama(prompt, model="gemma3:4b"):
    """
    Ollama API를 호출하여 답변을 생성합니다.
    실패 시 'Failed to generate response'를 반환하여
    app.py의 필터링 로직이 감지할 수 있도록 합니다.
    """
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }

    try:
        response = requests.post(url, json=payload, timeout=90) # 타임아웃 추가로 무한 대기 방지
        response.raise_for_status()

        # .get()을 사용하여 키가 없을 경우의 에러 방지
        result = response.json()
        return result.get('response', "Failed to generate response")

    except Exception as e:
        # 연결 오류나 HTTP 에러 발생 시 지정된 에러 문구 반환
        print(f"Ollama 커뮤니케이션 에러: {e}") # 로그 확인용
        return "Failed to generate response"

def ipcc_ai_analyst(question, vectorstore): # vectorstore 인자 추가
    # 1. 지식 검색 (상위 4개 조각 추출)
    docs = vectorstore.similarity_search(question, k=4)

    # 2. 컨텍스트 구성 및 출처 정리
    # .get()을 사용하여 메타데이터가 없을 경우 'Unknown'으로 처리
    context_list = []
    source_list = []

    for d in docs:
        section = d.metadata.get('section', 'Unknown')
        context_list.append(f"[Source: {section}]\n{d.page_content}")
        source_list.append(section)

    context = "\n\n".join(context_list)
    sources = list(set(source_list))

    # 3. IPCC 전용 프롬프트 설계
    prompt = f"""
너는 IPCC 6차 보고서(AR6) 전문 분석가야. 아래 제공된 [Context]를 바탕으로 질문에 답해줘.

[Context]
{context}

[Question]
{question}

[Guidelines]
- 반드시 제공된 Context의 내용을 근거로 답변할 것.
- 답변은 전문적이고 신뢰감 있는 한국어로 작성할 것.
- 인류의 활동에 의한 기후 변화의 책임을 명확히 전달할 것 (Action-oriented).
- 전문 용어는 영어 원문을 괄호 안에 병기할 것 (예: 탄소 중립(Net Zero)).
- 답변 마지막에 참고한 섹션들을 나열할 것.

답변:
"""

    # 4. Ollama 호출
    response = ask_ollama(prompt)

    # 5. 에러 체크 로직
    error_keywords = ["Failed to generate response", "답변 생성 실패", "Ollama 연결 오류", "에러 발생"]
    if any(keyword in response for keyword in error_keywords):
        return response, [], []

    return response, docs, sources

# --- 5. UI Layout & Main Logic ---
vectorstore = load_resources()

st.title("🌍 기후 위기 알리미: IPCC 보고서 요약봇")
st.info("JCCR 논문 기반 페르소나가 적용된 IPCC AR6 전문 분석 리포트를 제공합니다.")

if "messages" not in st.session_state:
    st.session_state.messages = []

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

if prompt := st.chat_input("질문을 입력하세요"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    if vectorstore:
        with st.chat_message("assistant"):
            with st.spinner("IPCC 보고서 분석 및 리포트 작성 중... "):
                response, docs, sources = ipcc_ai_analyst(prompt, vectorstore)

                if sources: # sources가 비어있지 않다는 것은 에러가 아니라는 뜻
                    st.markdown(f"### 📢 분석 결과 (참조: {', '.join(sources)})")
                    st.markdown(response)

                    with st.expander("📚 근거 자료 원문 및 페이지 확인"):
                        for d in docs:
                            section = d.metadata.get('section', 'AR6')
                            page_num = d.metadata.get('page', 0)
                            pdf_path = f"/content/drive/MyDrive/IPCC_data/IPCC_AR6_SYR_FullVolume_{section}.pdf"

                            col1, col2 = st.columns([1, 4])
                            with col1:
                                thumb = get_pdf_thumbnail(pdf_path, page_num)
                                if thumb: st.image(thumb, caption=f"Page {page_num+1}")
                                else: st.caption("미리보기 불가")
                            with col2:
                                st.markdown(f"**[{section}] Page {page_num+1}**")
                                st.write(d.page_content[:300] + "...")
                            st.divider()
                else:
                    st.error(response)
                    st.warning("⚠️ Ollama 서버 상태를 확인해 주세요. 서버가 꺼져 있으면 답변을 생성할 수 없습니다.")

                st.session_state.messages.append({"role": "assistant", "content": response})
                clear_memory()
    else:
        st.error("데이터베이스를 로드할 수 없습니다.")

## 서버 실행

In [ ]:
import subprocess
import time

# 1. Terminate existing Streamlit and Cloudflared processes
print("🧹 Cleaning up existing Streamlit and Cloudflared processes...")
subprocess.run(["pkill", "-f", "streamlit"])
subprocess.run(["pkill", "-f", "cloudflared"])
time.sleep(2)

# 2. Start the updated Streamlit application
print("🚀 Starting the Streamlit application on port 8501...")
log_file = "streamlit_restart_logs.txt"
with open(log_file, "w") as f:
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"], stdout=f, stderr=f)

# 3. Wait for initialization
time.sleep(5)
print(f"✅ Streamlit process initiated. Check {log_file} for details.")

# 4. Ollama 서버를 백그라운드에서 실행
# stdout/stderr를 무시하지 않고 연결해두면 나중에 로그 확인이 가능합니다.
process = subprocess.Popen(['ollama', 'serve'],
                           stdout=subprocess.PIPE,
                           stderr=subprocess.PIPE,
                           text=True)

# 5. 서버가 완전히 준비될 때까지 대기
print("Ollama 서버를 시작하는 중...")
time.sleep(10)
print("Ollama 서버가 준비되었습니다!")

# 6. 서버 실행
!cloudflared tunnel --url http://localhost:8501